# Using AI to Explain AI 🔍🤖
### Interpreting, Explaining, and Scoring Neurons

Large language models contain hundreds of thousands of **neurons**, and nobody hand-labels what each one does. A striking idea from interpretability research: **use one AI to explain another AI**.

This notebook walks through that idea in three movements, using a tiny, fully transparent *synthetic* model so everything runs in Colab with no API key.

**Learning objectives**
1. Explain how a neuron's role can be *hypothesized from the tokens that make it activate*.
2. Describe the **Subject → Explainer → Simulator** pipeline and what each model does.
3. Compute an **Explanation Score** by correlating a Simulator's predicted activations with a neuron's real activations — and reason about what high vs low scores mean.
4. Interactively probe synthetic neurons to feel *why most real neurons are hard to explain*.

**Storyline:** `interpret` → `explain` → `score`.

In [ ]:
# === Environment setup ===
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html
import hashlib

%matplotlib inline

# Detect Colab and enable the custom widget manager so ipywidgets render there.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    try:
        from google.colab import output as _colab_output
        _colab_output.enable_custom_widget_manager()
    except AttributeError:
        pass  # older Colab: the default widget manager still works
except ImportError:
    IN_COLAB = False

# Reproducibility: one shared Generator, plus legacy global seed.
SEED = 42
RNG = np.random.default_rng(SEED)
np.random.seed(SEED)

# --- sanity checks ---
assert RNG is not None and SEED == 42
assert all(m is not None for m in (np, plt, widgets))

print(f"Environment ready. numpy {np.__version__} | running in Colab: {IN_COLAB}")

## Concept 1 — A neuron's job shows up in *what makes it fire*

We can't read a neuron's mind, but we *can* watch it: feed in text and record its activation on each token. If a neuron reliably lights up on the words **right**, **correct**, and **true**, a reasonable guess is *"this is a correctness neuron."*

This is exactly how interpretability researchers surface candidate roles — for example, a much-cited **GPT-2 neuron (Layer 31, Neuron 892)** that activates on words about *doing things correctly*.

Below we build a miniature **Subject model** with a few neurons whose true roles we secretly know, then probe them token-by-token to watch this happen.

In [ ]:
# === Synthetic "Subject" model ===
# A few rule-based neurons, each with an interpretable ground-truth role, plus
# one deliberately uninterpretable "noisy" neuron.
import string

# Fixed vocabulary spanning the semantic groups our neurons respond to.
VOCAB = [
    # correctness words
    "right", "correct", "true", "yes", "valid",
    # number words
    "one", "two", "three", "ten", "hundred",
    # color words
    "red", "blue", "green", "yellow",
    # filler / neutral words
    "the", "a", "to", "of", "cat",
    "runs", "sky", "and", "is", "it",
]

_CORRECT_WORDS = {"right", "correct", "true", "yes", "valid"}
_NUMBER_WORDS = {"one", "two", "three", "ten", "hundred"}
_COLOR_WORDS = {"red", "blue", "green", "yellow"}

_NOISE_SALT = "ai-explains-ai::v1"


def _norm(tok):
    # Lowercase a token and strip surrounding punctuation/whitespace.
    return tok.lower().strip(string.punctuation + string.whitespace)


def correctness_rule(tok):
    return 1.0 if _norm(tok) in _CORRECT_WORDS else 0.0


def number_rule(tok):
    return 1.0 if _norm(tok) in _NUMBER_WORDS else 0.0


def color_rule(tok):
    return 1.0 if _norm(tok) in _COLOR_WORDS else 0.0


def noisy_rule(tok):
    # Deterministic pseudo-random activation in [0, 1].
    # Uses a salted hashlib digest (NOT Python's randomized builtin hash) so the
    # value is identical across kernels/processes -> the C12 aggregate is stable.
    digest = hashlib.md5((_NOISE_SALT + _norm(tok)).encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big") / 0xFFFFFFFF


# Ordered mapping: neuron_id -> {label, rule, group}. 'group' is the trigger set.
NEURONS = {
    "N_correct": {"label": "Correctness neuron (fires on right/correct/true)",
                  "rule": correctness_rule, "group": _CORRECT_WORDS},
    "N_number": {"label": "Number neuron (fires on one/two/three/...)",
                 "rule": number_rule, "group": _NUMBER_WORDS},
    "N_color": {"label": "Color neuron (fires on red/blue/green/...)",
                "rule": color_rule, "group": _COLOR_WORDS},
    "N_noisy": {"label": "Noisy neuron (no interpretable pattern)",
                "rule": noisy_rule, "group": None},
}


def activation(neuron_id, tokens):
    # Per-token activations for a neuron, as a float numpy array.
    if neuron_id not in NEURONS:
        raise KeyError(
            f"Unknown neuron_id {neuron_id!r}. Valid ids: {list(NEURONS)}")
    if isinstance(tokens, str):
        tokens = [tokens]
    try:
        tokens = list(tokens)
    except TypeError:
        raise TypeError("tokens must be a str or an iterable of str")
    if not all(isinstance(t, str) for t in tokens):
        raise TypeError("every token must be a str")
    rule = NEURONS[neuron_id]["rule"]
    return np.array([float(rule(t)) for t in tokens], dtype=float)


# --- sanity checks ---
_a = activation("N_correct", ["right", "cat", "correct"])
assert _a[0] == 1.0 and _a[1] == 0.0 and _a[2] == 1.0
assert np.allclose(activation("N_noisy", ["cat"]), activation("N_noisy", ["cat"]))
assert set(NEURONS) == {"N_correct", "N_number", "N_color", "N_noisy"}
assert activation("N_correct", []).shape == (0,)
print("Subject model ready. Neurons:", ", ".join(NEURONS))

In [ ]:
# === Interactive probe: which tokens make a neuron fire? ===
DEFAULT_SENTENCE = "the answer is right and correct"


def tokenize(sentence):
    # Whitespace split + punctuation strip; drop empty tokens.
    return [t for t in (_norm(w) for w in sentence.split()) if t]


def _activation_to_color(value):
    # Map an activation in [0, 1] to a white -> orange highlight CSS color.
    v = float(np.clip(value, 0.0, 1.0))
    r = 255
    g = int(round(255 + (140 - 255) * v))
    b = int(round(255 + (0 - 255) * v))
    return f"#{r:02x}{g:02x}{b:02x}"


def render_tokens(sentence, neuron_id):
    tokens = tokenize(sentence)
    if not tokens:
        return HTML("<i>Type a sentence to probe a neuron.</i>")
    acts = activation(neuron_id, tokens)
    label = NEURONS[neuron_id]["label"]
    spans = []
    for tok, act in zip(tokens, acts):
        a = float(np.clip(act, 0.0, 1.0))
        color = _activation_to_color(a)
        safe = html.escape(tok)
        spans.append(
            f'<span title="activation={a:.2f}" '
            f'style="background-color:{color};padding:2px 6px;margin:1px;'
            f'border-radius:4px;display:inline-block;">{safe}</span>'
        )
    legend = (f'<div style="margin-bottom:6px;font-family:sans-serif">'
              f'<b>Neuron:</b> {html.escape(label)} '
              f'<span style="color:#888;">(darker = stronger activation)</span></div>')
    return HTML(legend + " ".join(spans))


sentence_box = widgets.Text(
    value=DEFAULT_SENTENCE, description="Sentence:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "initial"},
)
neuron_dropdown = widgets.Dropdown(
    options=[(NEURONS[nid]["label"], nid) for nid in NEURONS],
    description="Neuron:",
    style={"description_width": "initial"},
)

widgets.interact(render_tokens, sentence=sentence_box, neuron_id=neuron_dropdown)

# --- sanity checks ---
assert tokenize("The answer is RIGHT.") == ["the", "answer", "is", "right"]
_html_obj = render_tokens("right cat", "N_correct")
assert hasattr(_html_obj, "data") and "right" in _html_obj.data

## Concept 2 — Subject → Explainer → Simulator

"Using AI to explain AI" is a three-model pipeline:

| Role | In the real paper | What it does |
|------|-------------------|--------------|
| **Subject** | GPT-2 | The model whose neuron we want to understand. We observe its activations. |
| **Explainer** | GPT-4 | Reads the Subject neuron's top-activating tokens and writes a short natural-language explanation. |
| **Simulator** | GPT-4 | Given *only the explanation* (not the neuron), predicts how strongly the neuron would fire on new tokens. |

The key constraint: the **Simulator never sees the real neuron** — only the Explainer's words. If the explanation is good, the Simulator's predictions should match reality.

> In this demo the Explainer and Simulator are **rule-based stand-ins** for GPT-4, so the notebook runs with no API key while preserving the method's logic.

In [ ]:
# === Rule-based Explainer and Simulator ===
# Explainer: sees only tokens + activations -> emits a natural-language guess.
# Simulator: sees only the explanation text -> predicts activations.
# The Simulator NEVER calls activation() or reads NEURONS -- that separation is
# exactly what makes the Explanation Score (Concept 3) meaningful.

SAMPLE_CORPUS = [
    "the answer is right and correct",
    "yes that is true and valid",
    "i counted one two three apples",
    "ten plus ninety is one hundred",
    "the sky is blue and the apple is red",
    "green and yellow are bright colors",
    "a cat runs to the door",
    "it is the end of the day",
]

# Theme word used when building / parsing explanation strings.
THEME_KEYWORDS = {
    "N_correct": "correctness",
    "N_number": "numbers",
    "N_color": "colors",
}
_THEME_TO_GROUP = {
    "correctness": _CORRECT_WORDS,
    "numbers": _NUMBER_WORDS,
    "colors": _COLOR_WORDS,
}


def explainer(neuron_id, corpus=None, top_k=5):
    # Summarize a neuron from its top-activating tokens over a corpus.
    if neuron_id not in NEURONS:
        raise KeyError(
            f"Unknown neuron_id {neuron_id!r}. Valid ids: {list(NEURONS)}")
    if corpus is None:
        corpus = SAMPLE_CORPUS
    # Aggregate the max activation per unique token.
    best = {}
    for sentence in corpus:
        toks = tokenize(sentence)
        acts = activation(neuron_id, toks)
        for tok, act in zip(toks, acts):
            best[tok] = max(best.get(tok, 0.0), float(act))
    ranked = sorted(best.items(), key=lambda kv: kv[1], reverse=True)
    top_tokens = [(t, s) for t, s in ranked if s > 0.0][:top_k]

    # A noisy/uninterpretable neuron gives a spread of FRACTIONAL activations;
    # interpretable neurons here fire crisply (0.0 or 1.0). Detect this first so
    # a coincidental overlap of top tokens with a group can't mislabel noise.
    fractional = sum(1 for v in best.values() if 0.001 < v < 0.999)
    looks_noisy = fractional >= 3

    # Infer a theme by matching top tokens against the known group sets.
    theme = None
    if top_tokens and not looks_noisy:
        top_set = {t for t, _ in top_tokens}
        for nid, kw in THEME_KEYWORDS.items():
            grp = _THEME_TO_GROUP[kw]
            if len(top_set & grp) >= 2 or (len(top_set) == 1 and top_set <= grp):
                theme = kw
                break

    if looks_noisy or not top_tokens:
        explanation = "No clear pattern (likely uninterpretable)."
    elif theme is not None:
        listed = ", ".join(t for t, _ in top_tokens[:3])
        explanation = f"This neuron fires on words about {theme}, especially: {listed}."
    else:
        listed = ", ".join(t for t, _ in top_tokens[:3])
        explanation = f"This neuron fires on words such as: {listed}."

    return {"neuron_id": neuron_id, "explanation": explanation,
            "top_tokens": top_tokens}


def _extract_tokens_from_explanation(explanation):
    # Recover the explicitly-listed trigger tokens from an explanation string.
    found = set()
    low = explanation.lower()
    if "especially:" in low:
        tail = low.split("especially:", 1)[1]
    elif "such as:" in low:
        tail = low.split("such as:", 1)[1]
    else:
        tail = ""
    for piece in tail.replace(".", " ").split(","):
        tok = _norm(piece)
        if tok:
            found.add(tok)
    return found


def simulator(explanation, tokens):
    # Predict per-token activations from the explanation text ALONE.
    # Never consults activation() or NEURONS -- only the explanation string.
    if isinstance(tokens, str):
        tokens = [tokens]
    tokens = list(tokens)
    listed = _extract_tokens_from_explanation(explanation)
    # Find a theme keyword mentioned in the explanation -> its synonym group.
    theme_group = set()
    low = explanation.lower()
    for kw, grp in _THEME_TO_GROUP.items():
        if kw in low:
            theme_group = grp
            break
    preds = []
    for tok in tokens:
        t = _norm(tok)
        if t in listed:
            preds.append(1.0)
        elif t in theme_group:
            preds.append(0.3)
        else:
            preds.append(0.0)
    return np.array(preds, dtype=float)


# --- sanity checks ---
_exp = explainer("N_correct")
assert any(t == "right" for t, _ in _exp["top_tokens"])
_e = explainer("N_correct")["explanation"]
assert simulator(_e, ["right", "cat"])[0] > simulator(_e, ["right", "cat"])[1]
_noisy = explainer("N_noisy")
assert "no clear pattern" in _noisy["explanation"].lower() or len(_noisy["top_tokens"]) <= 5
assert simulator("fires on words about correctness, especially: right", []).shape == (0,)
print("Explainer + Simulator ready.")

In [ ]:
# === Interactive Explainer demo ===
explainer_dropdown = widgets.Dropdown(
    options=[(NEURONS[nid]["label"], nid) for nid in NEURONS],
    description="Neuron:",
    style={"description_width": "initial"},
)


def show_explanation(neuron_id):
    try:
        exp = explainer(neuron_id)
    except Exception as err:  # readable message instead of a silent failure
        return HTML(f"<b style='color:red'>Error:</b> {html.escape(str(err))}")
    truth = NEURONS[neuron_id]["label"]
    parts = [
        "<div style='font-family:sans-serif'>",
        f"<p><b>Ground-truth role (hidden from the Explainer):</b> "
        f"{html.escape(truth)}</p>",
        f"<p><b>Explainer's guess (from activations only):</b><br>"
        f"<i>{html.escape(exp['explanation'])}</i></p>",
    ]
    if exp["top_tokens"]:
        rows = "".join(
            f"<tr><td style='padding:2px 12px'>{html.escape(t)}</td>"
            f"<td style='padding:2px 12px'>{s:.2f}</td></tr>"
            for t, s in exp["top_tokens"]
        )
        parts.append(
            "<table style='border-collapse:collapse'>"
            "<tr><th style='text-align:left;padding:2px 12px'>top token</th>"
            "<th style='text-align:left;padding:2px 12px'>activation</th></tr>"
            f"{rows}</table>"
        )
    else:
        parts.append("<p>No strongly-activating tokens found.</p>")
    parts.append("</div>")
    return HTML("".join(parts))


widgets.interact(show_explanation, neuron_id=explainer_dropdown)

# --- sanity checks ---
_h = show_explanation("N_color")
assert "blue" in _h.data.lower() or "color" in _h.data.lower()
assert all(show_explanation(n) is not None for n in NEURONS)

## Concept 3 — The Explanation Score

How do we know if an explanation is *good*? We score it:

1. Ask the **Simulator** to predict the neuron's activation on each token, using only the explanation.
2. Get the **Subject's real** activations on the same tokens.
3. Take the **correlation** between predicted and real, mapped to a score in **[0, 1]**.

For example, with the explanation *"fires on running-related words"* the Simulator might predict `running → 10`, `to → 0`. If the real neuron agrees, the score is high; if not, it's low.

A high score means the explanation truly captures the neuron's behavior. Let's compute it.

In [ ]:
# === The Explanation Score function ===
def explanation_score(neuron_id, sentence, corpus=None):
    # Correlate Simulator predictions with the Subject's real activations.
    # Returns a score in [0, 1]: 1.0 = perfect agreement, 0.5 = uninformative
    # (degenerate / constant inputs), 0.0 = perfectly anti-correlated.
    if corpus is None:
        corpus = SAMPLE_CORPUS
    tokens = tokenize(sentence)
    real = activation(neuron_id, tokens)
    exp = explainer(neuron_id, corpus)
    pred = simulator(exp["explanation"], tokens)

    # Fewer than 2 tokens -> not enough points to correlate -> uninformative.
    if len(tokens) < 2:
        return 0.5
    # Constant array(s) -> correlation undefined. Equal constants => agreement.
    if np.std(real) == 0 or np.std(pred) == 0:
        return 1.0 if np.allclose(real, pred) else 0.5

    r = np.corrcoef(real, pred)[0, 1]
    if np.isnan(r):
        return 0.5
    return float(np.clip((r + 1.0) / 2.0, 0.0, 1.0))


# --- sanity checks ---
_s = explanation_score("N_correct", "the answer is right and correct")
assert 0.0 <= _s <= 1.0 and _s > 0.6
assert explanation_score("N_noisy", "one two right blue cat runs") <= \
    explanation_score("N_correct", "one two right blue cat runs")
assert explanation_score("N_correct", "cat") == 0.5
assert not np.isnan(explanation_score("N_color", "the sky is blue"))
print("Explanation Score function ready.")

In [ ]:
# === Interactive scoring demo: real vs simulated activations ===
score_dropdown = widgets.Dropdown(
    options=[(NEURONS[nid]["label"], nid) for nid in NEURONS],
    description="Neuron:",
    style={"description_width": "initial"},
)
score_sentence_box = widgets.Text(
    value="running fast is the right thing to do",
    description="Sentence:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "initial"},
)
score_out = widgets.Output()


def score_and_plot(neuron_id, sentence):
    with score_out:
        clear_output(wait=True)  # flicker-free redraw, avoids figure pile-up
        try:
            tokens = tokenize(sentence)
            if not tokens:
                print("Type a sentence to score a neuron.")
                return
            MAX_TOK = 20
            truncated = len(tokens) > MAX_TOK
            shown = tokens[:MAX_TOK]
            real = activation(neuron_id, shown)
            exp = explainer(neuron_id)
            pred = simulator(exp["explanation"], shown)
            score = explanation_score(neuron_id, sentence)

            x = np.arange(len(shown))
            width = 0.4
            fig, ax = plt.subplots(figsize=(min(12, 1.5 + 0.8 * len(shown)), 4))
            ax.bar(x - width / 2, real, width, label="Subject (real)", color="#1f77b4")
            ax.bar(x + width / 2, pred, width, label="Simulator (predicted)", color="#ff7f0e")
            ax.set_xticks(x)
            ax.set_xticklabels(shown, rotation=45, ha="right")
            ax.set_ylim(0, 1.05)
            ax.set_ylabel("activation")
            ax.set_title(
                f"{NEURONS[neuron_id]['label']}  |  Explanation Score = {score:.2f}")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)

            print(f"Explanation: {exp['explanation']}")
            print(f"Explanation Score: {score:.2f}")
            if truncated:
                print(f"(showing first {MAX_TOK} of {len(tokens)} tokens)")
        except Exception as err:
            print(f"Could not score/plot: {err}")


display(score_out)
widgets.interact(score_and_plot, neuron_id=score_dropdown, sentence=score_sentence_box)

# --- sanity checks ---
score_and_plot("N_correct", "the right and correct answer")
score_and_plot("N_noisy", "cat runs to the sky")

In [ ]:
# === Aggregate view: average Explanation Score per neuron ===
def compute_average_scores(corpus=None):
    if corpus is None:
        corpus = SAMPLE_CORPUS
    sentences = [s for s in corpus if tokenize(s)]
    if not sentences:
        print("Corpus empty -- falling back to a small default corpus.")
        sentences = ["the answer is right", "one two three", "the sky is blue"]
    per_neuron = {}
    for nid in NEURONS:
        scores = [explanation_score(nid, s, sentences) for s in sentences]
        scores = [0.5 if np.isnan(v) else v for v in scores]
        per_neuron[nid] = float(np.mean(scores)) if scores else 0.5
    return per_neuron


per_neuron = compute_average_scores()
overall_avg = float(np.mean(list(per_neuron.values())))

labels = [NEURONS[nid]["label"].split(" (")[0] for nid in per_neuron]
heights = [per_neuron[nid] for nid in per_neuron]
colors = ["#d62728" if nid == "N_noisy" else "#1f77b4" for nid in per_neuron]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(labels, heights, color=colors)
ax.axhline(overall_avg, color="green", linestyle="--",
           label=f"overall avg = {overall_avg:.2f}")
ax.axhline(0.15, color="gray", linestyle=":",
           label="real GPT-2 avg (~0.15)")
ax.set_ylim(0, 1.05)
ax.set_ylabel("avg Explanation Score")
ax.set_title("Average Explanation Score per synthetic neuron")
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
plt.show()
plt.close(fig)

print("Per-neuron average Explanation Score:")
for nid, v in per_neuron.items():
    print(f"  {nid:10s} {v:.3f}  ({NEURONS[nid]['label']})")
print(f"Overall average: {overall_avg:.3f}")
print("Takeaway: the uninterpretable 'noisy' neuron scores far lower than the "
      "interpretable ones -- most real neurons look like the noisy one.")

# --- sanity checks ---
_avg = compute_average_scores()
assert set(_avg) == set(NEURONS) and all(0.0 <= v <= 1.0 for v in _avg.values())
assert _avg["N_noisy"] == min(_avg.values())
assert 0.0 <= float(np.mean(list(_avg.values()))) <= 1.0

## Wrap-up — what this tells us about real models

Our interpretable neurons score high; the **noisy neuron scores near chance**. That mirrors reality: when OpenAI scored *every* GPT-2 neuron this way, the average explanation score was only about **0.15** for the GPT-4 explainer (versus **~0.18** for human explainers). Most neurons are simply *hard to explain*.

**Why scores stay low — limitations**
- **Weak Simulator:** if the Simulator can't role-play the neuron well, even a perfect explanation scores poorly — the score is capped by the Simulator.
- **Explainer/Simulator collusion:** using the same model family for both can inflate or distort scores.
- **Data-selection bias:** explanations built only from top-activating tokens may miss the neuron's full behavior.
- **Polysemanticity:** a single neuron may not *have* one language-expressible function — it can encode several unrelated features at once.

**Extensions to try**
- Swap the synthetic Subject for a real **GPT-2** via 🤗 `transformers` and read genuine activations.
- Replace the rule-based Explainer/Simulator with real LLM calls.
- Explore *features* from sparse autoencoders, which are often more interpretable than raw neurons.

You've now walked the full **interpret → explain → score** loop that lets AI help explain AI. 🎉